# Exercise 1: Identifying Data, Information, and Knowledge - Solutions

In [1]:
%%capture
%pip install notebook duckdb jupysql matplotlib duckdb-engine;

In [ ]:
%%capture
%config SqlMagic.autopandas = True;
%config SqlMagic.feedback = False;
%config SqlMagic.displaycon = False;
import duckdb;
import os;
%load_ext sql
conn = duckdb.connect();
%sql conn --alias duckdb
%sql duckdb:///:memory:
%sql CREATE TABLE raw_survey AS SELECT * FROM '../data/survey_results_public.parquet';



#### Task: Fill in the blank after GROUP BY below to start. Note, in DuckDB SQL, you can GROUP BY aliases.

In [46]:
%%sql
SELECT
    STRING_SPLIT(LanguageHaveWorkedWith, ';') AS Language,
    MEDIAN(CAST(ConvertedCompYearly AS INT)) AS MedSalary
FROM
    raw_survey
WHERE
    LanguageHaveWorkedWith IS NOT NULL
    AND ConvertedCompYearly IS NOT NULL 
    AND ConvertedCompYearly != 'NA'
    --Language is the solution
    GROUP BY Language
LIMIT 20;

,Language,MedSalary
0,"[Ada, Assembly, Bash/Shell (all shells), C, C#...",109000.0
1,"[Bash/Shell (all shells), HTML/CSS, JavaScript...",87475.0
2,"[HTML/CSS, JavaScript, Ruby, TypeScript]",94846.0
3,"[C, C++, Python, Rust]",60000.0
4,"[C#, JavaScript, Kotlin, PowerShell, SQL, Type...",120000.0
5,"[Bash/Shell (all shells), C, C++, Fortran, Pyt...",84110.5
6,"[Bash/Shell (all shells), C, C#, C++, HTML/CSS...",2900.0
7,"[Bash/Shell (all shells), HTML/CSS, JavaScript...",91018.0
8,"[C#, Erlang, Java, PowerShell, Python, SQL, Ty...",19983.0
9,"[Bash/Shell (all shells), HTML/CSS, Java, Java...",75877.0


#### Task: Modify the query from Part B to filter for results where Country is 'United States of America' and YearsCodingExp is between 5 and 10 years.

In [47]:
%%sql
SELECT
    STRING_SPLIT(LanguageHaveWorkedWith, ';') AS Language,
    MEDIAN(CAST(ConvertedCompYearly AS INT)) AS MedSalary,
    CAST(YearsCode AS INT) AS YearsCodingExp
FROM
    raw_survey
WHERE
    LanguageHaveWorkedWith IS NOT NULL
    AND ConvertedCompYearly IS NOT NULL
    AND ConvertedCompYearly != 'NA'
    AND YearsCode != 'NA'
    -- BETWEEN 5 AND 10 is the solution
    -- Country = 'United States of America' is the solution
    AND YearsCodingExp BETWEEN 5 AND 10
    AND Country = 'United States of America'
GROUP BY
    Language,
    YearsCodingExp
ORDER BY
    YearsCodingExp DESC
LIMIT 100;

,Language,MedSalary,YearsCodingExp
0,"[C++, PowerShell, Python]",120000.0,10
1,"[Bash/Shell (all shells), C, C++, HTML/CSS, Ja...",120000.0,10
2,"[Bash/Shell (all shells), C++, Go, HTML/CSS, J...",91000.0,10
3,"[Bash/Shell (all shells), C#, Java, PowerShell...",85000.0,10
4,"[Bash/Shell (all shells), C++, Python]",160000.0,10
...,...,...,...
95,"[Bash/Shell (all shells), C#, C++, HTML/CSS, J...",40000.0,10
96,"[Bash/Shell (all shells), JavaScript, PowerShe...",150000.0,10
97,"[Bash/Shell (all shells), Go, HTML/CSS, JavaSc...",190.0,10
98,"[C, HTML/CSS, JavaScript, Python, Rust, SQL, T...",190000.0,10


#### Task: Replace blanks with the correct conditional statements for the CASE expression, and the WHERE clause.  

In [48]:
%%sql
SELECT
    STRING_SPLIT(LanguageHaveWorkedWith, ';') AS Language,
    MEDIAN(CAST(ConvertedCompYearly AS INT)) AS MedSalary,
    CAST(YearsCode AS INT) AS YearsCodingExp,
    CASE WHEN YearsCodingExp >= 0 AND YearsCodingExp <= 10 THEN ' 0 - 10'
        WHEN YearsCodingExp > 10 AND YearsCodingExp <= 25 THEN '10+ - 25'
        -- YearsCodingExp > 25 AND YearsCodingExp <= 50 THEN '50+ - 100' is the solution
        WHEN YearsCodingExp > 25 AND YearsCodingExp <= 50 THEN '50+ - 100'
    END AS YearsCodingExpBucket
FROM
    raw_survey
WHERE
    LanguageHaveWorkedWith IS NOT NULL
    AND ConvertedCompYearly IS NOT NULL
    AND ConvertedCompYearly != 'NA'
    AND YearsCode != 'NA'
    -- BETWEEN 5 AND 10 is the solution
    -- Country = 'United States of America' is the solution
    AND YearsCodingExp BETWEEN 5 AND 10
    AND Country = 'United States of America'
GROUP BY
    Language,
    YearsCodingExp
ORDER BY
    YearsCodingExp DESC
LIMIT 100;

,Language,MedSalary,YearsCodingExp,YearsCodingExpBucket
0,"[Bash/Shell (all shells), HTML/CSS, Java, Java...",143000.0,10,0 - 10
1,"[Bash/Shell (all shells), C, C++, Python]",150000.0,10,0 - 10
2,"[Go, HTML/CSS, Java, JavaScript, Python, Ruby,...",215000.0,10,0 - 10
3,"[Bash/Shell (all shells), Python, Rust]",500000.0,10,0 - 10
4,"[Bash/Shell (all shells), C, C++, HTML/CSS, Po...",105966.0,10,0 - 10
...,...,...,...,...
95,[NA],134000.0,10,0 - 10
96,"[Bash/Shell (all shells), Go, HTML/CSS, JavaSc...",185000.0,10,0 - 10
97,"[HTML/CSS, JavaScript, Python, SQL]",110000.0,10,0 - 10
98,"[HTML/CSS, JavaScript, Python, R, Rust, TypeSc...",94000.0,10,0 - 10


#### Task: Fill in the blanks as part of the last piece of the WHERE clause to have only Rust users appear in the filter. Do the same afterwards, but for C++ users. 

In [49]:
%%sql
SELECT
    'Rust' AS Language,
    MEDIAN(CAST(ConvertedCompYearly AS INT)) AS MedSalary,
    CASE WHEN CAST(YearsCode AS INT)  >= 0 AND CAST(YearsCode AS INT)  <= 10 THEN ' 0 - 10'
        WHEN CAST(YearsCode AS INT)  > 10 AND CAST(YearsCode AS INT)  <= 25 THEN '10+ - 25'
        WHEN CAST(YearsCode AS INT)  > 25 AND CAST(YearsCode AS INT)  <= 50 THEN '50+ - 100'
    END AS YearsCodingExpBucket
FROM
    raw_survey
WHERE
    LanguageHaveWorkedWith IS NOT NULL
    AND ConvertedCompYearly IS NOT NULL
    AND ConvertedCompYearly != 'NA'
    AND YearsCode != 'NA'
    AND YearsCodingExpBucket != 'None'
    AND Country = 'United States of America'
    -- 'Rust' IN STRING_SPLIT(LanguageHaveWorkedWith, ';') is the solution
    AND 'Rust' IN STRING_SPLIT(LanguageHaveWorkedWith, ';') 
GROUP BY
    Language,
    YearsCodingExpBucket
ORDER BY
    YearsCodingExpBucket DESC
LIMIT 100;

,Language,MedSalary,YearsCodingExpBucket
0,Rust,200000.0,50+ - 100
1,Rust,170000.0,10+ - 25
2,Rust,91000.0,0 - 10


In [50]:
%%sql
SELECT
    'C++' AS Language,
    MEDIAN(CAST(ConvertedCompYearly AS INT)) AS MedSalary,
    CASE WHEN CAST(YearsCode AS INT)  >= 0 AND CAST(YearsCode AS INT)  <= 10 THEN ' 0 - 10'
        WHEN CAST(YearsCode AS INT)  > 10 AND CAST(YearsCode AS INT)  <= 25 THEN '10+ - 25'
        WHEN CAST(YearsCode AS INT)  > 25 AND CAST(YearsCode AS INT)  <= 50 THEN '50+ - 100'
    END AS YearsCodingExpBucket
FROM
    raw_survey
WHERE
    LanguageHaveWorkedWith IS NOT NULL
    AND ConvertedCompYearly IS NOT NULL
    AND ConvertedCompYearly != 'NA'
    AND YearsCode != 'NA'
    AND YearsCodingExpBucket != 'None'
    AND Country = 'United States of America'
    --'C++' IN STRING_SPLIT(LanguageHaveWorkedWith, ';') is the solution
    --AND 'C++' IN STRING_SPLIT(LanguageHaveWorkedWith, ';') 
GROUP BY
    Language,
    YearsCodingExpBucket
ORDER BY
    YearsCodingExpBucket DESC
LIMIT 100;

,Language,MedSalary,YearsCodingExpBucket
0,C++,174000.0,50+ - 100
1,C++,154000.0,10+ - 25
2,C++,100000.0,0 - 10


#### Task: Fill in the blanks as part of the last piece of the WHERE clause to have only Rust users AND C++ users appear in the filter. 

In [51]:
%%sql
SELECT
    'C++ and Rust' AS Language,
    MEDIAN(CAST(ConvertedCompYearly AS INT)) AS MedSalary,
    CASE WHEN CAST(YearsCode AS INT)  >= 0 AND CAST(YearsCode AS INT)  <= 10 THEN ' 0 - 10'
        WHEN CAST(YearsCode AS INT)  > 10 AND CAST(YearsCode AS INT)  <= 25 THEN '10+ - 25'
        WHEN CAST(YearsCode AS INT)  > 25 AND CAST(YearsCode AS INT)  <= 50 THEN '50+ - 100'
    END AS YearsCodingExpBucket
FROM
    raw_survey
WHERE
    LanguageHaveWorkedWith IS NOT NULL
    AND ConvertedCompYearly IS NOT NULL
    AND ConvertedCompYearly != 'NA'
    AND YearsCode != 'NA'
    AND YearsCodingExpBucket != 'None'
    AND Country = 'United States of America'
    --'C++' IN STRING_SPLIT(LanguageHaveWorkedWith, ';') is the solution
    AND 'C++' IN STRING_SPLIT(LanguageHaveWorkedWith, ';') 
    -- 'Rust' IN STRING_SPLIT(LanguageHaveWorkedWith, ';') is the solution
    AND 'Rust' IN STRING_SPLIT(LanguageHaveWorkedWith, ';') 
GROUP BY
    Language,
    YearsCodingExpBucket
ORDER BY
    YearsCodingExpBucket DESC
LIMIT 100;

,Language,MedSalary,YearsCodingExpBucket
0,C++ and Rust,206000.0,50+ - 100
1,C++ and Rust,150000.0,10+ - 25
2,C++ and Rust,80000.0,0 - 10


#### Task: Fill in the blanks as part of the last piece of the WHERE clause to have both PostgreSQL users OR MongoDB users appear in the filter. 

In [52]:
%%sql
SELECT
    CASE WHEN 'MongoDB' IN STRING_SPLIT(DatabaseHaveWorkedWith, ';') AND 'PostgreSQL' IN STRING_SPLIT(DatabaseHaveWorkedWith, ';') THEN 'MongoDB & PostgreSQL'
         WHEN 'MongoDB' IN STRING_SPLIT(DatabaseHaveWorkedWith, ';') THEN 'MongoDB'
         WHEN 'PostgreSQL' IN STRING_SPLIT(DatabaseHaveWorkedWith, ';') THEN 'PostgreSQL' END AS DatabaseExp,
    MEDIAN(CAST(ConvertedCompYearly AS INT)) AS MedSalary,
    CASE WHEN CAST(YearsCode AS INT)  >= 0 AND CAST(YearsCode AS INT)  <= 10 THEN ' 0 - 10'
        WHEN CAST(YearsCode AS INT)  > 10 AND CAST(YearsCode AS INT)  <= 25 THEN '10+ - 25'
        WHEN CAST(YearsCode AS INT)  > 25 AND CAST(YearsCode AS INT)  <= 50 THEN '50+ - 100'
    END AS YearsCodingExpBucket
FROM
    raw_survey
WHERE
    DatabaseExp IS NOT NULL
    AND ConvertedCompYearly IS NOT NULL
    AND ConvertedCompYearly != 'NA'
    AND YearsCode != 'NA'
    AND YearsCodingExpBucket != 'None'
    AND Country = 'United States of America'
    -- ('MongoDB' IN DatabaseExp OR 'PostgreSQL' IN DatabaseExp) is the solution
    AND ('MongoDB' IN DatabaseExp OR 'PostgreSQL' IN DatabaseExp)
GROUP BY
    DatabaseExp,
    YearsCodingExpBucket
ORDER BY
    DatabaseExp, YearsCodingExpBucket DESC
LIMIT 100;

,DatabaseExp,MedSalary,YearsCodingExpBucket
0,MongoDB,165000.0,50+ - 100
1,MongoDB,150000.0,10+ - 25
2,MongoDB,95750.0,0 - 10
3,MongoDB & PostgreSQL,185000.0,50+ - 100
4,MongoDB & PostgreSQL,170000.0,10+ - 25
5,MongoDB & PostgreSQL,122000.0,0 - 10
6,PostgreSQL,184500.0,50+ - 100
7,PostgreSQL,168000.0,10+ - 25
8,PostgreSQL,112000.0,0 - 10


## Standout: 
What are some potential pitfalls of this type of analysis? We believe we have found some patterns we know to be true, but how could we have erred in coming to these conclusions?

#### This is freeform answer. 